**Импорт необходимых библиотек**

In [84]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, RandomizedSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from xgboost import XGBClassifier

**Baseline предобработка**

In [19]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

log_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_scaled = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Parch', 'Pclass']),
    ('log', log_pipeline, ['Fare']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

**Менее требовательная предобработка для архитектур на деревьях**

In [20]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

log_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_tree = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Parch', 'Pclass']),
    ('log', log_pipeline, ['Fare']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

**Оценка качества**

In [21]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

**Загрузим данные**

In [22]:
df = pd.read_csv('E:/ML/titanic-ml/data/raw/train.csv')
y = df['Survived']
X = df.drop(columns='Survived')

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.2,
    shuffle=True,
    random_state=42,
    stratify=y
)

**Рассмотрим следующие модели:**
* Logistic Regression + L1/L2
* Decision Tree
* kNN
* SVM
* Random Forest
* Gradient Boosting
* XGBoost

**Инициализируем модели**

In [23]:
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', LogisticRegression())
    ]),

    'kNN': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', KNeighborsClassifier())
    ]),

    'SVM': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', SVC())
    ]),

    'Decision Tree': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', DecisionTreeClassifier())
    ]),

    'Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', RandomForestClassifier())
    ]),

    'Gradient Boosting': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', GradientBoostingClassifier())
    ]),

    'XGBoost': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', XGBClassifier())
    ])
}

**Функция для оценки**

In [24]:
def evaluate_models(models, X, y, cv, scoring):
    results = []

    for name, model in models.items():
        cv_results = cross_validate(
            model,
            X,
            y,
            cv=cv,
            scoring=scoring
        )

        result = {'Model': name}

        for metric in scoring:
            scores = cv_results[f'test_{metric}']

            result[f'{metric}_mean'] = scores.mean()
            result[f'{metric}_std'] = scores.std()

        results.append(result)

    return pd.DataFrame(results)

results = evaluate_models(models, X_train, y_train, cv, scoring)

**Отсортируем модели по accuracy**

In [26]:
results.sort_values(
    'accuracy_mean',
    ascending=False
).round(3)

,Model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
4,Random Forest,0.769,0.060,0.717,0.111,0.675,0.162,0.683,0.104,0.796,0.077
0,Logistic Regression,0.769,0.081,0.714,0.116,0.644,0.188,0.669,0.140,0.829,0.105
2,SVM,0.758,0.048,0.723,0.104,0.633,0.157,0.658,0.088,0.813,0.072
5,Gradient Boosting,0.758,0.082,0.702,0.143,0.659,0.109,0.676,0.109,0.804,0.068
1,kNN,0.741,0.065,0.692,0.124,0.601,0.139,0.635,0.100,0.790,0.073
6,XGBoost,0.735,0.088,0.692,0.189,0.631,0.189,0.636,0.134,0.769,0.071
3,Decision Tree,0.724,0.057,0.664,0.124,0.615,0.113,0.628,0.078,0.697,0.060


Из лучших моделей с базовыми настройками выберем 3 и проведем эксперемент с настройками гиперпараметров и предобработкой

Модели для детального анализа: **Random Forest, Logistic Regression, SVM**

In [28]:
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', LogisticRegression())
    ]),

    'SVM': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', SVC())
    ]),

    'Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', RandomForestClassifier())
    ]),
}

results = evaluate_models(models, X_train, y_train, cv, scoring)
results

,Model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,Logistic Regression,0.768889,0.080883,0.713625,0.115884,0.643956,0.187761,0.668745,0.139918,0.829096,0.105466
1,SVM,0.757937,0.048135,0.723095,0.103695,0.632967,0.157047,0.657550,0.087566,0.812612,0.071938
2,Random Forest,0.746190,0.083073,0.687305,0.144643,0.628571,0.175019,0.645665,0.133848,0.792882,0.068458


Рассмотрим **Logistic Regression**

**Найдем оптимпльные праметры с использованием Grif Search**

Создадим сетку 

In [30]:
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__class_weight": [None, "balanced"]
}

Поиск

In [33]:
grid = GridSearchCV(
    models['Logistic Regression'],
    param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print(grid.best_params_)
print(f"{grid.best_score_:.3f}")

{'classifier__C': 0.1, 'classifier__class_weight': 'balanced'}
0.774


Рассмотрим **SVM**

Создадим сетку для Random Search

In [34]:
param_dist = {
    "classifier__C": np.logspace(-3,3,20),
    "classifier__gamma": ["scale", "auto"],
    "classifier__kernel":["rbf", "linear"]
}

Поиск

In [37]:
search = RandomizedSearchCV(
    models['SVM'],
    param_distributions=param_dist,
    n_iter=30,
    cv=cv,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print(search.best_params_)
print(f"{search.best_score_:.3f}")

{'classifier__kernel': 'linear', 'classifier__gamma': 'auto', 'classifier__C': np.float64(0.1623776739188721)}
0.791


Рассмотрим **Random Forest**

Создадим сетку для Random Search

In [53]:
rf_params = {
    "classifier__n_estimators": [50, 100, 200, 300],
    "classifier__max_depth": [2, 3, 5],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 3, 5],
}

Поиск

In [54]:
search_rf = RandomizedSearchCV(
    models['Random Forest'],
    rf_params,
    n_iter=50,
    cv=cv,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)


search_rf.fit(
    X_train,
    y_train
)

print(search_rf.best_params_)
print(f"{search_rf.best_score_:.3f}")

{'classifier__n_estimators': 300, 'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 1, 'classifier__max_depth': 5}
0.752


**SVM** показал лучший результат 

|Модель|accuracy|Параметры|
|-|-|-|
|**Logistic Regression**|0.774|'classifier__C': 0.1, 'classifier__class_weight': 'balanced'|
|**SVM**|**0.791**|'classifier__kernel': 'linear', 'classifier__gamma': 'auto', 'classifier__C': np.float64(0.1623776739188721)|
|**Random Forest**|0.752|{'classifier__n_estimators': 300, 'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 1, 'classifier__max_depth': 5}|

Рассмотрим, что можно сделать с обработкой данных, чтобы улучшить качество

In [73]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

log_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_scaled = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Parch', 'Pclass']),
    ('log', log_pipeline, ['Fare']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model SVM': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', SVC(
            kernel='linear',
            gamma='auto',
            C=0.1623776739188721
        ))
    ])
}

results_base = evaluate_models(models, X_train, y_train, cv, scoring)
results_base

,Model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,Best model SVM,0.79127,0.090704,0.750659,0.130937,0.658242,0.166961,0.698681,0.148134,0.824101,0.099132


**Рассмотрим различные подходы к подготовке данных для лучшей модели**

**Другая стратегия заполнения Age - mean**

In [74]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

log_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_scaled = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Parch', 'Pclass']),
    ('log', log_pipeline, ['Fare']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model SVM': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', SVC(
            kernel='linear',
            gamma='auto',
            C=0.1623776739188721
        ))
    ])
}

results = evaluate_models(models, X_train, y_train, cv, scoring)

print(f"\nПрирост accuracy: {results['accuracy_mean'].values[0] - results_base['accuracy_mean'].values[0]:.3f}")


Прирост accuracy: -0.017


Ухудшение качества

**Уберем логарифмирование параметра Fare**

In [75]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

log_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_scaled = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Parch', 'Pclass', 'Fare']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model SVM': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', SVC(
            kernel='linear',
            gamma='auto',
            C=0.1623776739188721
        ))
    ])
}
results = evaluate_models(models, X_train, y_train, cv, scoring)

print(f"\nПрирост accuracy: {results['accuracy_mean'].values[0] - results_base['accuracy_mean'].values[0]:.3f}")


Прирост accuracy: -0.045


ухудшение

**добвим логарифмирование параметру SibSp**

In [72]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

log_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_scaled = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'Parch', 'Pclass']),
    ('log', log_pipeline, ['Fare',  'SibSp']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model SVM': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', SVC(
            kernel='linear',
            gamma='auto',
            C=0.1623776739188721
        ))
    ])
}

results = evaluate_models(models, X_train, y_train, cv, scoring)

print(f"\nПрирост accuracy: {results['accuracy_mean'].values[0] - results_base['accuracy_mean'].values[0]:.3f}")


Прирост accuracy: -0.011


Нет прироста accuracy

**Добавим логарифмирование Parch**

In [77]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

log_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_scaled = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Pclass']),
    ('log', log_pipeline, ['Fare',  'Parch']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model SVM': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', SVC(
            kernel='linear',
            gamma='auto',
            C=0.1623776739188721
        ))
    ])
}

results = evaluate_models(models, X_train, y_train, cv, scoring)

print(f"\nПрирост accuracy: {results['accuracy_mean'].values[0] - results_base['accuracy_mean'].values[0]:.3f}")


Прирост accuracy: -0.017


Нет прироста accuracy

**добвим логарифмирование параметру Age**

In [81]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

log_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_scaled = ColumnTransformer([
    ('num', num_pipeline, ['Parch', 'SibSp', 'Pclass']),
    ('log', log_pipeline, ['Fare','Age']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model SVM': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', SVC(
            kernel='linear',
            gamma='auto',
            C=0.1623776739188721
        ))
    ])
}

results = evaluate_models(models, X_train, y_train, cv, scoring)

print(f"\nПрирост accuracy: {results['accuracy_mean'].values[0] - results_base['accuracy_mean'].values[0]:.3f}")


Прирост accuracy: -0.017


Нет прироста accuracy

**Обработаем Pclass как категориальный признак**

In [85]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

log_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p))
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_scaled = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Parch']),
    ('log', log_pipeline, ['Fare']),
    ('cat', cat_pipeline, ['Sex', 'Embarked', 'Pclass'])
])

models = {
    'Best model SVM': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', SVC(
            kernel='linear',
            gamma='auto',
            C=0.1623776739188721
        ))
    ])
}

results = evaluate_models(models, X_train, y_train, cv, scoring)

print(f"\nПрирост accuracy: {results['accuracy_mean'].values[0] - results_base['accuracy_mean'].values[0]:.3f}")


Прирост accuracy: 0.000


нет изменений

**Использование степенного преобразование (Yeo-Johnson) вместо логарифмирования**

In [86]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

power_pipeline = Pipeline([
    ('power_tr', PowerTransformer(method='yeo-johnson')),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_scaled = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Parch']),
    ('power', power_pipeline, ['Fare']),
    ('cat', cat_pipeline, ['Sex', 'Embarked', 'Pclass'])
])

models = {
    'Best model SVM': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', SVC(
            kernel='linear',
            gamma='auto',
            C=0.1623776739188721
        ))
    ])
}

results = evaluate_models(models, X_train, y_train, cv, scoring)

print(f"\nПрирост accuracy: {results['accuracy_mean'].values[0] - results_base['accuracy_mean'].values[0]:.3f}")


Прирост accuracy: -0.006


нет прироста

Итоговый вариант

In [88]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

log_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor_scaled = ColumnTransformer([
    ('num', num_pipeline, ['Age', 'SibSp', 'Parch', 'Pclass']),
    ('log', log_pipeline, ['Fare']),
    ('cat', cat_pipeline, ['Sex', 'Embarked'])
])

models = {
    'Best model SVM': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', SVC(
            kernel='linear',
            gamma='auto',
            C=0.1623776739188721
        ))
    ])
}

results_base = evaluate_models(models, X_train, y_train, cv, scoring)
results_base

,Model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,Best model SVM,0.79127,0.090704,0.750659,0.130937,0.658242,0.166961,0.698681,0.148134,0.824101,0.099132


Зафиксируем модель

In [89]:
best_model = Pipeline([
    ('preprocessor', preprocessor_scaled),
    ('classifier', SVC(
        kernel='linear',
        gamma='auto',
        C=0.1623776739188721
    ))
])